In [1]:
# MA416 Final Project Sprint 1 CNN
# Jacob Richardson, Kevin Cotellesso
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


In [ ]:
# -------------- NETWORK SETTINGS -------------- #
# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = False

# --- Hyperparameters ---
num_epochs = 10
batch_size = 32
learning_rate = 0.001

# --- Data transformations ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]) 
 
# --- Datasets and DataLoaders ---
dataset_root = "../Datasets/SmellySongs9k/spectrograms"


In [15]:

# --- Load dataset ---
full_dataset = datasets.ImageFolder(root=dataset_root, transform=transform)

#change gpu used to available gpu
torch.cuda.set_device(1)

dataset_size = len(full_dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

torch.manual_seed(42)
train_data, val_data = torch.utils.data.random_split(full_dataset, [train_size, val_size])

class_names = full_dataset.classes


train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=pin_memory)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=pin_memory)

print(f"Loaded {dataset_size} images from {dataset_root} ({len(class_names)} classes). Train/Val = {train_size}/{val_size}")

# --- Load pretrained VGG16 ---
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# --- Freeze convolutional base ---
for param in model.features.parameters():
    param.requires_grad = False

# --- Replace the classifier (for 2 output classes) ---
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 2)

model = model.to(device)

# --- Loss and optimizer ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

Loaded 23108 images from ../Datasets/SmellySongs23K/spectrograms (2 classes). Train/Val = 18487/4621


In [ ]:
from PIL import Image
import torch.nn.functional as F

#create a cnn for classifying grayscale spectrogram images. Do not use vvgg16, create a small cnn from scratch with pytorch
class SpectrogramCNN(nn.Module):
    def __init__(self):
        super(SpectrogramCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, 2)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 56 * 56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
        
model = SpectrogramCNN().to(device)


In [18]:
#train model
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / train_size
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/10], Loss: 0.7009
Epoch [2/10], Loss: 0.7009
Epoch [3/10], Loss: 0.7009


KeyboardInterrupt: 